In [ ]:
#!pip install python-dotenv

In [ ]:
import requests
from dotenv import load_dotenv
import os
load_dotenv()
BASE = "https://api.themoviedb.org/3"
KEY = os.getenv("TMDB_KEY")
print(f"Key loaded: {KEY[:5]}...")
IMG = "https://image.tmdb.org/t/p/w300"

GENRE_MAP = {
    "Action":28, "Comedy":35, "Drama":18, "Thriller":53,
    "Sci-Fi":878, "Romance":10749, "Animation":16,
    "Documentary":99, "Horror":27, "Adventure":12
}

def search_movies(genres, min_rating=6.0, decade=None, page=1):
    genre_ids = ",".join(str(GENRE_MAP[g]) for g in genres if g in GENRE_MAP)
    params = {
        "api_key": KEY, "with_genres": genre_ids,
        "vote_average.gte": min_rating, "vote_count.gte": 100,
        "sort_by": "popularity.desc", "page": page
    }
    if decade:
        params["primary_release_date.gte"] = f"{decade}-01-01"
        params["primary_release_date.lte"] = f"{decade+9}-12-31"
    r = requests.get(f"{BASE}/discover/movie", params=params)
    return r.json().get("results", [])[:20]

def get_movie_details(tmdb_id):
    r = requests.get(f"{BASE}/movie/{tmdb_id}",
        params={"api_key": KEY, "append_to_response": "credits"})
    d = r.json()
    cast = [c["name"] for c in d.get("credits",{}).get("cast",[])[:5]]
    crew = d.get("credits",{}).get("crew",[])
    director = next((c["name"] for c in crew if c["job"]=="Director"), "Unknown")
    return {
        "id": d["id"], "title": d["title"],
        "year": d.get("release_date","")[:4],
        "rating": round(d.get("vote_average", 0), 1),
        "genres": [g["name"] for g in d.get("genres",[])],
        "overview": d.get("overview",""),
        "poster": IMG + d["poster_path"] if d.get("poster_path") else None,
        "cast": cast, "director": director,
        "runtime": d.get("runtime", 0)
    }

In [ ]:
# Test 1 — search movies
results = search_movies(["Action", "Thriller"], min_rating=7.0)
for m in results[:3]:
    print(f"{m['title']} ({m['release_date'][:4]}) — Rating: {m['vote_average']}")

In [ ]:
# Test 2 — get full details
movie_id = results[0]["id"]
details = get_movie_details(movie_id)
print(f"Title: {details['title']}")
print(f"Year: {details['year']}")
print(f"Rating: {details['rating']}")
print(f"Director: {details['director']}")
print(f"Cast: {', '.join(details['cast'])}")
print(f"Poster: {details['poster']}")